# 05. 리포트 생성 & 가중치 튜닝 (Report & Tuning)
## 서울 성동구 요식 가맹점 조기 경보 시스템 | 빅콘테스트 2025

> **STEP 12**: 분기별 위험 등급 분류 + LLM 기반 자연어 리포트  
> **STEP 13**: 피처 가중치 3단계 최적화 (Mann-Whitney → Grid → Bayesian)


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

snap  = pd.read_csv('../p_project_snapshot.csv', encoding='utf-8-sig')
panel = pd.read_csv('../p_project_features.csv', encoding='utf-8-sig')

FEAT_ALL = [c for c in snap.columns if c.startswith('dw_') or c.startswith('f_')]
print(f'snap shape: {snap.shape}')
print(f'Columns with risk_rank: {[c for c in snap.columns if "rank" in c.lower()]}')


### STEP 12-1. 위험 등급 분류

In [ ]:
# 피처 그룹 정의 (STEP 13에서도 사용)
FEAT_INTERNAL    = ['f_sales_lvl','f_trx_lvl','f_spend_lvl','f_sales_trend','f_trx_trend',
                    'f_return_rate','f_return_trend','f_float_ratio','f_float_trend','f_resid_ratio']
FEAT_COMPETITIVE = ['f_rank_ind','f_rank_dist','f_rank_ind_trend','f_rank_dist_trend',
                    'f_vs_ind_sales','f_vs_ind_trend']
FEAT_EXTERNAL    = ['f_peer_close_ind','f_peer_close_dist']
FEAT_ALL         = FEAT_INTERNAL + FEAT_COMPETITIVE + FEAT_EXTERNAL

# ================================================================
# STEP 12: 분기별 리포트 + LLM 기반 해석 시스템
# ================================================================
# 점포별 위험 수준 분류 -> 자연어 리포트 자동 생성
# Claude API 연동 구조 (API 키 없어도 템플릿 기반 리포트 생성 가능)

# 12-1. 위험 등급 분류
def classify_risk(rank_pct):
    if pd.isna(rank_pct): return '평가불가'
    if rank_pct >= 85:    return '위험'
    if rank_pct >= 65:    return '경고'
    if rank_pct >= 40:    return '주의'
    return '정상'

snap['risk_level'] = snap['risk_rank_pct'].apply(classify_risk)
print('위험 등급 분포:')
print(snap['risk_level'].value_counts().to_string())
print()
print('폐업 점포(is_closed_obs=1)의 위험 등급:')
print(snap[snap['is_closed_obs']==1]['risk_level'].value_counts().to_string())

### STEP 12-2. 리포트 생성 함수 (FEAT_KOR, get_top_risks, generate_report)

In [ ]:
# 12-2. 주요 리스크 요인 텍스트 매핑
FEAT_KOR = {
    'f_sales_trend':    '매출액 버킷 악화 추세',
    'f_trx_trend':      '거래건수 버킷 악화 추세',
    'f_vs_ind_trend':   '업종 평균 대비 매출 하락',
    'f_rank_ind_trend': '업종 내 매출 순위 하락',
    'f_rank_dist_trend':'상권 내 매출 순위 하락',
    'f_return_rate':    '재방문 고객 비율 낮음',
    'f_return_trend':   '재방문율 감소 추세',
    'f_float_ratio':    '유동 고객 의존도 높음',
    'f_float_trend':    '유동 고객 비율 증가',
    'f_resid_ratio':    '상주 고객 비율 낮음',
    'f_sales_lvl':      '현재 매출 수준 낮음',
    'f_rank_ind':       '업종 내 하위 순위',
    'f_peer_close_ind': '동종 업종 폐업률 높음',
    'f_peer_close_dist':'주변 상권 폐업률 높음',
}

RANK_INT  = ['rank_'+f for f in FEAT_INTERNAL]
RANK_COMP = ['rank_'+f for f in FEAT_COMPETITIVE]
RANK_EXT  = ['rank_'+f for f in FEAT_EXTERNAL]

def get_top_risks(row, n=3):
    scores = {}
    for feat in FEAT_ALL:
        rcol = 'rank_' + feat
        if rcol in row.index and pd.notna(row[rcol]):
            scores[feat] = row[rcol]
    top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]
    return [(FEAT_KOR.get(f, f), v) for f, v in top]

# 12-3. 점포별 자연어 리포트 생성 함수
def generate_report(row):
    level = row['risk_level']
    rank  = row.get('risk_rank_pct', np.nan)
    ind   = row.get('HPSN_MCT_BZN_CD_NM', '(업종 미상)')
    top3  = get_top_risks(row)

    level_msg = {
        '위험': '업종 내 상위 15% 위험 구간에 해당합니다. 즉각적인 경영 개선 조치가 필요합니다.',
        '경고': '업종 내 상위 35% 위험 구간입니다. 주요 지표를 면밀히 모니터링하세요.',
        '주의': '업종 내 상위 60% 위험 구간입니다. 개선 여지가 있는 지표를 점검하세요.',
        '정상': '업종 내 하위 40% 위험 구간으로 양호한 상태입니다.',
        '평가불가': '비교 가능한 동종 업종 데이터가 부족하여 순위 평가가 어렵습니다.',
    }.get(level, '')

    rank_str = f'{rank:.0f}%' if pd.notna(rank) else 'N/A'
    top3_str = '\n'.join([f'    {i+1}. {name} (업종 내 상위 {v:.0f}% 위험)'
                          for i, (name, v) in enumerate(top3)])

    report = f"""
=== 분기별 경영 위기 조기 경보 리포트 ===
점포명:     {row.get('MCT_NM', 'N/A')}
업종:       {ind}
평가기간:   2024년 4분기 (2024.10~12 기준)

[위험 등급]  {level}  (업종 내 위험 순위: {rank_str})
{level_msg}

[주요 위험 신호 Top 3]
{top3_str}

[LLM 조언 요청 프롬프트 예시]
  "성동구 {ind} 업종 점포입니다. 최근 3개월간 {top3[0][0] if top3 else '매출 변화'}가 
  감지되었습니다. 소상공인 관점에서 단기/중기 개선 방안을 구체적으로 제안해주세요."
"""
    return report

### STEP 12-3. 고위험 점포 리포트 출력 & 전체 저장

In [ ]:
# 12-4. 예시: 고위험 점포 리포트 출력
high_risk = snap[snap['risk_level']=='위험'].sort_values('risk_rank_pct', ascending=False)
if len(high_risk) > 0:
    sample = high_risk.iloc[0]
    print(generate_report(sample))

# 12-5. 전체 리포트 CSV 저장
snap['report_text'] = snap.apply(generate_report, axis=1)
snap[['ENCODED_MCT','MCT_NM','HPSN_MCT_BZN_CD_NM','risk_level','risk_rank_pct',
      'risk_score','is_closed_obs','report_text']].to_csv(
    '../p_project_reports.csv', index=False, encoding='utf-8-sig')
print(f'[SAVED] p_project_reports.csv  ({len(snap)}개 점포 리포트)')
print()
print('위험 등급별 점포 수:')
for level in ['위험','경고','주의','정상','평가불가']:
    cnt = (snap['risk_level']==level).sum()
    print(f'  {level}: {cnt}개')


---
## STEP 13. 가중치 튜닝 (3단계 체계적 최적화)

### STAGE 1. Mann-Whitney Effect-size 피처 가중치

#### 1a. 데이터 준비 & Mann-Whitney Effect Size 계산

In [ ]:
# ================================================================
# STEP 13: 가중치 튜닝 (3단계 체계적 최적화)
# ================================================================
# 튜닝 대상:
#   1. 피처 가중치  — Mann-Whitney effect size 기반 (데이터 주도)
#   2. Decay lambda — 최근 데이터 강조 정도 (λ 작을수록 최신 강조)
#   3. Component 가중치 — internal / competitive / external 비중 최적화
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import itertools, time

if 'panel' not in dir():
    panel = pd.read_csv('../p_project_features.csv', encoding='utf-8-sig')
    snap  = pd.read_csv('../p_project_snapshot.csv', encoding='utf-8-sig')

FEAT_INTERNAL    = ['f_sales_lvl','f_trx_lvl','f_spend_lvl','f_sales_trend','f_trx_trend',
                    'f_return_rate','f_return_trend','f_float_ratio','f_float_trend','f_resid_ratio']
FEAT_COMPETITIVE = ['f_rank_ind','f_rank_dist','f_rank_ind_trend','f_rank_dist_trend',
                    'f_vs_ind_sales','f_vs_ind_trend']
FEAT_EXTERNAL    = ['f_peer_close_ind','f_peer_close_dist']
FEAT_ALL         = FEAT_INTERNAL + FEAT_COMPETITIVE + FEAT_EXTERNAL
MIN_GROUP        = 5

y = snap['is_closed_obs'].values
base_auc = roc_auc_score(y, snap['risk_score'].fillna(50))
print(f'[BASE] risk_score AUC: {base_auc:.4f}')
print()

# ── STAGE 1: Effect-size 기반 피처 가중치 ─────────────────────
print('=' * 58)
print('STAGE 1: Mann-Whitney Effect-size 피처 가중치')
print('=' * 58)

eff = {}
for f in FEAT_ALL:
    c = snap[snap['is_closed_obs']==1]['dw_'+f].dropna()
    a = snap[snap['is_closed_obs']==0]['dw_'+f].dropna()
    if len(c) < 3 or len(a) < 3:
        eff[f] = 0.0; continue
    stat, _ = stats.mannwhitneyu(c, a, alternative='greater')
    eff[f] = max(0.0, (2*stat)/(len(c)*len(a)) - 1)   # clip to 0 if hurts

#### 1b. 피처 가중치 정규화 & 출력

In [ ]:
def norm_weights(feat_list, eff_dict):
    vals = np.array([eff_dict.get(f, 0) for f in feat_list])
    total = vals.sum()
    if total == 0: return {f: 1/len(feat_list) for f in feat_list}
    return dict(zip(feat_list, vals/total))

w_int  = norm_weights(FEAT_INTERNAL,    eff)
w_comp = norm_weights(FEAT_COMPETITIVE, eff)
w_ext  = norm_weights(FEAT_EXTERNAL,    eff)

print(f'  Internal 피처 가중치 (top 5):')
for f, w in sorted(w_int.items(), key=lambda x: x[1], reverse=True)[:5]:
    bar = '#' * int(w*50)
    print(f'    {f:<28}: {w:.3f} {bar}')
print(f'  Competitive 피처 가중치:')
for f, w in sorted(w_comp.items(), key=lambda x: x[1], reverse=True):
    bar = '#' * int(w*50)
    print(f'    {f:<28}: {w:.3f} {bar}')

# Effect-weighted 스코어 계산 (기존 rank_ 컬럼 활용)
def ew_score(row, feat_list, weights):
    vals, ws = [], []
    for f in feat_list:
        v = row.get('rank_'+f, np.nan)
        if pd.notna(v):
            vals.append(v); ws.append(weights[f])
    if not vals: return np.nan
    return np.average(vals, weights=ws)

#### 1c. Effect-Weighted 점수 적용 & AUC 확인

In [ ]:

snap['score_int_ew']  = snap.apply(lambda r: ew_score(r, FEAT_INTERNAL,    w_int),  axis=1)
snap['score_comp_ew'] = snap.apply(lambda r: ew_score(r, FEAT_COMPETITIVE,  w_comp), axis=1)
snap['score_ext_ew']  = snap.apply(lambda r: ew_score(r, FEAT_EXTERNAL,     w_ext),  axis=1)

auc_ew = roc_auc_score(y, (0.5*snap['score_int_ew'] + 0.3*snap['score_comp_ew'] +
                            0.2*snap['score_ext_ew']).fillna(50))
print(f'\n  Effect-weighted score AUC (기존 가중치 유지): {auc_ew:.4f}')

### STAGE 2. Decay Lambda 최적화

#### 2a. Lambda 후보별 AUC / Lift@5% 계산

In [ ]:
# ── STAGE 2: Decay Lambda 최적화 ──────────────────────────────
print()
print('=' * 58)
print('STAGE 2: Decay Lambda 최적화')
print('=' * 58)

def decay_wmean(vals, lam):
    v = vals[~np.isnan(vals)]
    if len(v) == 0: return np.nan
    w = np.array([lam**(len(v)-1-i) for i in range(len(v))])
    return float(np.average(v, weights=w))

lambda_results = []
for lam in [0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00]:
    rows = []
    for mct_id, grp in panel.groupby('ENCODED_MCT', sort=False):
        grp  = grp.sort_values('TA_YM')
        base = {'ENCODED_MCT': mct_id,
                'HPSN_MCT_BZN_CD_NM': grp['HPSN_MCT_BZN_CD_NM'].iat[0],
                'is_closed_obs': int(grp['is_closed_obs'].iat[0])}
        for f in FEAT_ALL:
            base['dw_'+f] = decay_wmean(grp[f].values, lam)
        rows.append(base)
    df_lam = pd.DataFrame(rows)

    # Effect-weighted percentile rank within industry
    ind_sz = df_lam.groupby('HPSN_MCT_BZN_CD_NM')['ENCODED_MCT'].transform('count')
    for f in FEAT_ALL:
        ranked = df_lam.groupby('HPSN_MCT_BZN_CD_NM')['dw_'+f].transform(
            lambda x: x.rank(pct=True, na_option='keep') * 100)
        df_lam['rank_'+f] = np.where(ind_sz >= MIN_GROUP, ranked, np.nan)

    s_int  = df_lam.apply(lambda r: ew_score(r, FEAT_INTERNAL,    w_int),  axis=1)
    s_comp = df_lam.apply(lambda r: ew_score(r, FEAT_COMPETITIVE,  w_comp), axis=1)
    s_ext  = df_lam.apply(lambda r: ew_score(r, FEAT_EXTERNAL,     w_ext),  axis=1)
    score  = (0.5*s_int + 0.3*s_comp + 0.2*s_ext).fillna(50)

    y_lam = df_lam['is_closed_obs'].values
    auc_l = roc_auc_score(y_lam, score)
    k5 = max(1, int(len(y_lam)*0.05))
    top5 = np.argsort(score.values)[-k5:]
    lift5 = y_lam[top5].mean() / y_lam.mean()
    lambda_results.append({'lambda': lam, 'auc': auc_l, 'lift5': lift5})
    bar = '#' * int((auc_l-0.6)*300)
    print(f'  lambda={lam:.2f}: AUC={auc_l:.4f}  Lift@5%={lift5:.1f}x  {bar}')

lam_df = pd.DataFrame(lambda_results)

#### 2b. 최적 Lambda 선택 & 스냅샷 재계산

In [ ]:
best_lam = lam_df.loc[lam_df['auc'].idxmax(), 'lambda']
print(f'\n  최적 lambda = {best_lam}')

# 최적 lambda로 dw_ 피처 재계산
print(f'  lambda={best_lam}으로 스냅샷 재계산 중...')
rows = []
for mct_id, grp in panel.groupby('ENCODED_MCT', sort=False):
    grp  = grp.sort_values('TA_YM')
    base = {'ENCODED_MCT': mct_id,
            'MCT_NM': grp['MCT_NM'].iat[0],
            'HPSN_MCT_BZN_CD_NM': grp['HPSN_MCT_BZN_CD_NM'].iat[0],
            'HPSN_MCT_ZCD_NM': grp['HPSN_MCT_ZCD_NM'].iat[0],
            'HPSN_MCT_ZCD_NM_1': grp['HPSN_MCT_ZCD_NM_1'].iat[0],
            'is_closed_obs': int(grp['is_closed_obs'].iat[0]),
            'is_closed_all': int(grp['is_closed_all'].iat[0]),
            'n_obs_months': int(grp['TA_YM'].notna().sum())}
    for f in FEAT_ALL:
        base['dw_opt_'+f] = decay_wmean(grp[f].values, best_lam)
    rows.append(base)

snap_opt = pd.DataFrame(rows)
ind_sz = snap_opt.groupby('HPSN_MCT_BZN_CD_NM')['ENCODED_MCT'].transform('count')
for f in FEAT_ALL:
    snap_opt['dw_'+f] = snap_opt['dw_opt_'+f]
    ranked = snap_opt.groupby('HPSN_MCT_BZN_CD_NM')['dw_opt_'+f].transform(
        lambda x: x.rank(pct=True, na_option='keep')*100)
    snap_opt['rank_'+f] = np.where(ind_sz >= MIN_GROUP, ranked, np.nan)

### STAGE 3. Component Weight 그리드 서치

In [ ]:
# ── STAGE 3: Component Weight 그리드 서치 ─────────────────────
print()
print('=' * 58)
print('STAGE 3: Component Weight 그리드 서치 (effect-weighted + 최적 lambda)')
print('=' * 58)

snap_opt['s_int']  = snap_opt.apply(lambda r: ew_score(r, FEAT_INTERNAL,    w_int),  axis=1)
snap_opt['s_comp'] = snap_opt.apply(lambda r: ew_score(r, FEAT_COMPETITIVE,  w_comp), axis=1)
snap_opt['s_ext']  = snap_opt.apply(lambda r: ew_score(r, FEAT_EXTERNAL,     w_ext),  axis=1)

y_opt  = snap_opt['is_closed_obs'].values
grid   = []
for wi in np.arange(0.40, 0.91, 0.05):
    for wc in np.arange(0.05, 0.41, 0.05):
        we = round(1.0 - wi - wc, 3)
        if we < 0.02 or we > 0.35: continue
        score = wi*snap_opt['s_int'] + wc*snap_opt['s_comp'] + we*snap_opt['s_ext']
        score = score.fillna(50)
        auc   = roc_auc_score(y_opt, score)
        k5    = max(1, int(len(y_opt)*0.05))
        top5  = np.argsort(score.values)[-k5:]
        lift5 = y_opt[top5].mean() / y_opt.mean()
        grid.append({'wi':round(wi,2),'wc':round(wc,2),'we':we,'auc':auc,'lift5':lift5})

grid_df = pd.DataFrame(grid).sort_values('auc', ascending=False)
print('  Top 5 조합 (AUC 기준):')
print(f'  {"w_int":>6} {"w_comp":>6} {"w_ext":>6} {"AUC":>7} {"Lift@5%":>8}')
for _, r in grid_df.head(5).iterrows():
    print(f'  {r.wi:>6.2f} {r.wc:>6.2f} {r.we:>6.2f} {r.auc:>7.4f} {r.lift5:>8.1f}x')

best_row = grid_df.iloc[0]
W_OPT = (best_row.wi, best_row.wc, best_row.we)
print(f'\n  최적: internal={W_OPT[0]}, competitive={W_OPT[1]}, external={W_OPT[2]}')

### STAGE 4. 최종 성능 비교 & 저장

#### 4a. 최종 최적화 점수 계산 (risk_score_opt / risk_rank_opt)

In [ ]:
# ── STAGE 4: 최종 최적화 스코어 & 성능 비교 ───────────────────
print()
print('=' * 58)
print('STAGE 4: 성능 비교 (before vs after 튜닝)')
print('=' * 58)

snap_opt['risk_score_opt'] = (
    W_OPT[0]*snap_opt['s_int'] +
    W_OPT[1]*snap_opt['s_comp'] +
    W_OPT[2]*snap_opt['s_ext']
).fillna(50)

# 업종 내 최종 순위
snap_opt['risk_rank_opt'] = np.where(
    ind_sz >= MIN_GROUP,
    snap_opt.groupby('HPSN_MCT_BZN_CD_NM')['risk_score_opt']
            .transform(lambda x: x.rank(pct=True, na_option='keep')*100),
    np.nan
)

#### 4b. 성능 비교 (BASE vs TUNED) & 저장

In [ ]:
def eval_metrics(y_true, score, label):
    score = np.array(score)
    auc = roc_auc_score(y_true, score)
    n, base = len(y_true), y_true.mean()
    metrics = {'label': label, 'auc': auc}
    for kp in [0.10, 0.05]:
        k    = max(1, int(n*kp))
        topk = np.argsort(score)[-k:]
        prec = y_true[topk].mean()
        metrics[f'lift_{int(kp*100)}pct'] = prec/base
        metrics[f'prec_{int(kp*100)}pct'] = prec*100
    return metrics

comparisons = [
    eval_metrics(y, snap['risk_score'].fillna(50).values,
                 'BASE  (lambda=0.85, equal weights, 0.5/0.3/0.2)'),
    eval_metrics(y_opt, snap_opt['risk_score_opt'].values,
                 f'TUNED (lambda={best_lam}, eff-size, {W_OPT[0]}/{W_OPT[1]}/{W_OPT[2]})'),
]

print(f'  {"설정":<46} {"AUC":>6} {"Lift@10%":>9} {"Lift@5%":>9}')
print('  ' + '-'*73)
for m in comparisons:
    print(f'  {m["label"]:<46} {m["auc"]:>6.4f} {m["lift_10pct"]:>9.1f}x {m["lift_5pct"]:>9.1f}x')

# 폐업 점포 순위 분포 비교
print()
closed_mask = y_opt == 1
c_base = snap[snap['is_closed_obs']==1]['risk_rank_pct'].dropna()
c_opt  = snap_opt[snap_opt['is_closed_obs']==1]['risk_rank_opt'].dropna()
print(f'  폐업 점포 risk_rank (업종 내 %) — 높을수록 위험 구간')
print(f'  BASE : mean={c_base.mean():.1f}%, median={c_base.median():.1f}%')
print(f'  TUNED: mean={c_opt.mean():.1f}%, median={c_opt.median():.1f}%')

# 최적 가중치 요약 저장
tuning_summary = {
    'best_lambda': best_lam,
    'w_internal': W_OPT[0],
    'w_competitive': W_OPT[1],
    'w_external': W_OPT[2],
    'feat_weights_internal': w_int,
    'feat_weights_competitive': w_comp,
    'feat_weights_external': w_ext,
    'base_auc': base_auc,
    'tuned_auc': comparisons[1]['auc'],
    'auc_gain': comparisons[1]['auc'] - base_auc,
}
print()
print(f'  AUC 개선: {tuning_summary["base_auc"]:.4f} -> {tuning_summary["tuned_auc"]:.4f}  (+{tuning_summary["auc_gain"]:+.4f})')

snap_opt.to_csv('../p_project_snapshot_tuned.csv', index=False, encoding='utf-8-sig')
print('[SAVED] p_project_snapshot_tuned.csv')
